#### How to run activity / pipeline task for looped setuops like list of NB's or List of Catagories as shown in below example 
- so the whole idea is pass value as list to next activity & have "for each" activity in between
- this for each activity will call notebbok or task in parallel for each value in list . 
- How values are passes from list to next NB ? simple - using widgets. 
- In job setting of next NB we under parameter section of job config , simple say for this Widget name - take value of passed from For Each activity. 
- See *Looped_processing* for example 

In [0]:
%sql
use catalog ecommerce;
create schema if not exists 
 demo1;

In [0]:
import re 

dbutils.widgets.text("wdg_cat_name", "")

run_for_cat = dbutils.widgets.get("wdg_cat_name")

main_table = spark.read.table("ecommerce.bronze.brz_category")

safe_cat = re.sub(r'[^a-zA-Z0-9_]', '_', run_for_cat)

print('category --> ', safe_cat)
    
df = spark.sql(f"""SELECT * FROM ecommerce.bronze.brz_category 
                   WHERE category_name = '{safe_cat}'
                   """)
    
df.write.format("delta").mode("overwrite").partitionBy("category_name").saveAsTable(f"ecommerce.demo.category_{safe_cat}_new")
    
print(f"Partitioned table for category '{safe_cat}' saved.")



In [0]:
%skip
import re

all_cats = dbutils.jobs.taskValues.get(
    taskKey="01_get_dist_cats",
    key= "cat_names",
    debugValue= ""
)

all_cats = all_cats.split(",")
print('all_Cats', all_cats)

main_table = spark.read.table("ecommerce.bronze.brz_category")

for each_cat in all_cats:
  

    safe_cat = re.sub(r'[^a-zA-Z0-9_]', '_', each_cat)
    print('category --> ', each_cat)
    
    df = spark.sql(f"""SELECT * FROM ecommerce.bronze.brz_category 
                   WHERE category_name = '{each_cat}'
                   """)
    
    df.write.format("delta").mode("overwrite").partitionBy("category_name").saveAsTable(f"ecommerce.demo.category_{safe_cat}")
    
    print(f"Partitioned table for category '{each_cat}' saved.")

